In [1]:
def build_judge_prompt(
    question: str,
    reasoning: str,
    evidence: str,
    search_count: int,
    max_search_limit: int
):
    return f"""
You are a Judge evaluating whether an AI agent has sufficient information to answer a question.

Question:
{question}

Current reasoning:
{reasoning}

Retrieved evidence:
{evidence}

Search count: {search_count}/{max_search_limit}

Decide ONE of the following:
- ACCEPT: The answer can be finalized with current evidence.
- CONTINUE_SEARCH: More information is needed.
- REFINE_QUERY: The search query was ineffective or misdirected.
- REFINE_EXTRACTION: Relevant evidence exists but was not properly used.

Return your decision in JSON format:
{{
  "decision": "<ONE_OPTION>",
  "rationale": "<short explanation>"
}}
"""


In [2]:
def run_judge(
    llm,
    tokenizer,
    question,
    reasoning,
    evidence,
    search_count,
    max_search_limit,
    max_tokens=256
):
    prompt = build_judge_prompt(
        question, reasoning, evidence, search_count, max_search_limit
    )

    chat_prompt = tokenizer.apply_chat_template(
        [{"role": "user", "content": prompt}],
        tokenize=False,
        add_generation_prompt=True
    )

    output = llm.generate(
        [chat_prompt],
        SamplingParams(
            max_tokens=max_tokens,
            temperature=0.0,
            top_p=1.0
        )
    )

    raw = output[0].outputs[0].text.strip()

    try:
        result = json.loads(raw)
        return result["decision"], result.get("rationale", "")
    except Exception:
        return "CONTINUE_SEARCH", "Judge output malformed"
